# Fig. 4A–D: Discovery of novel viruses from the 41k RNA-seq data set

Prepare viral annotation tables for the external validation samples.
Run the companion scripts at the indicated steps.


In [1]:
import os
from collections import defaultdict
import pandas as pd
import numpy as np
from Bio import SeqIO
import random

In [2]:
# Define paths
basedir = "/path/to/project"

# Output directory
path_out = f"{basedir}/output/revise_final/Fig4/"

# Input files
# BLASTx output directory
path_in = f"{basedir}/data/250821_3883samples_external_downloaded251110/250821_3883samples_external/blastout_all_protein/"
# Contig FASTA files
path_in_fasta = f"{basedir}/data/250821_3883samples_external_downloaded251110/250821_3883samples_external/contig_viral_hit/"
# Sample information
sradata_path = f"{basedir}/data/data_external_251110/sample_list_cuttlefish_minia_diamond_add_positive0to2_blastx_all.txt"
srameta_path = f"{basedir}/data/data_external_251110/Aves_Mam_mbio_metadata_ISG_nologan3942_ML_251110.txt"
# Blacklisted viral families
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
# BLASTn results
path_in_blastn = f"{basedir}/data/251111_external/virus_hit_all_contigs_external.txt"

if not os.path.exists(path_out):
    os.makedirs(path_out)
print("saving files:", path_out)


saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig4/


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """
    Collect taxids from BLASTx top hits across samples.
    """
    # Initialize the taxid set
    taxids = set()

    # Read BLASTx hits from each sample
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            # Read BLAST hits
            col_name = [
                "qseqid", "sseqid", "pident", "length", "mismatch",
                "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
                "bitscore", "qframe", "staxids", "stitle", "qlen", 
                "slen", "qcovhsp", "scovhsp"
            ]

            if os.path.getsize(full_path) == 0:
                print("file size 0 ")
                continue

            one_sample = pd.read_table(full_path, header=None, dtype={13:str})  # Read taxids as strings
            one_sample.columns = col_name

            # Check that every hit has a valid bit score
            if one_sample['bitscore'].isna().sum() > 0:
                print("NaN", filename)
            if (one_sample['bitscore'].dtype != np.float64) and (one_sample['bitscore'].dtype != int):
                print("dtype", filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])

            # Sort hits by bit score
            one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

            # Remove hits without taxids
            one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

            # Keep the top hit for each query
            one_sample_uniq = one_sample_sored.groupby("qseqid").first()

            # Collect taxids
            for line_taxid in one_sample_uniq["staxids"].to_list():
                if (line_taxid is None) or (line_taxid!=line_taxid):
                    continue
                raw_taxids = line_taxid.split(";")
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])

    # Write one taxid per line
    with open(takonkit_path, "w") as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + "\n")
    print(f"{len(taxids)} taxids in total")

    return taxids


def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """
    Annotate top hits and retain non-phage, non-blacklisted viral contigs.
    """
    # Count hits spanning multiple taxonomic groups
    num_ambiguous_hit = 0

    # Assign the confusion-matrix group
    cm_group = 'external'

    # Output file path
    output_folder = out_dir + "/taxonomy_output/"
    if not(os.path.exists(output_folder)):
        os.mkdir(output_folder)

    output_all_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.taxonomy.txt"
    output_viral_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.txt"
    output_viral_black_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.black.txt"

    # Read BLAST hits
    col_name = [
        "qseqid", "sseqid", "pident", "length", "mismatch",
        "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
        "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",
        "qcovhsp", "scovhsp"
    ]

    one_sample = pd.read_table(input_file, header=None, dtype={13:str})
    one_sample.columns = col_name

    # Add the sample ID
    one_sample["file_name"] = input_file.split("/")[-1].split(".")[0].split("_")[0]

    # Sort hits by bit score
    one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

    # Remove hits without taxids
    one_sample_sored = one_sample_sored.dropna(subset=["staxids"])

    # Keep the top hit for each query
    one_sample_uniq = one_sample_sored.groupby("qseqid").first()

    # Count all contigs
    num_all_hits = len(set(one_sample_uniq.index.to_list()))

    # Add taxonomic annotations
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax["staxids_lst"] = one_sample_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
    one_sample_tax["name_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
    one_sample_tax["group_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

    # Check for mixed viral and non-viral taxonomic groups
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, "group_lst"]
        if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1

    one_sample_tax["taxid_blastx"] = one_sample_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  
    one_sample_tax["family"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "family"] if((x is not None) and (x==x)) else x)
    one_sample_tax["genus"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "genus"] if((x is not None) and (x==x)) else x)
    one_sample_tax["species"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "species"] if((x is not None) and (x==x)) else x)

    # Mark hits as viral only when all taxonomic groups are viral
    one_sample_tax["is_virus"] = one_sample_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

    # Retain viral hits
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus["is_virus"] == 1]

    # Count viral contigs
    num_viral_hits = len(set(one_sample_virus.index.to_list()))

    # Read blacklisted viral families
    blackvirus = []
    with open(blacklst, "r") as f_in:
        for line in f_in:
            blackvirus.append(line.strip())

    # Exclude blacklisted viral families
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed["family"].isin(blackvirus)]

    # Count viral contigs after blacklist filtering
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))

    # Save results
    one_sample_tax.to_csv(output_all_path, sep="\t")
    one_sample_virus.to_csv(output_viral_path, sep="\t")
    one_sample_black_removed.to_csv(output_viral_black_path, sep="\t")

    return one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit


def count_viral_contigs(path_in: str, path_out, lineage_info, blacklist_path, infection_th=1):
    """
    Collect viral contigs and summarize detection by confusion-matrix group.
    """
    # Initialize summaries for each confusion-matrix group
    cm_hits = defaultdict(list)
    cm_props = defaultdict(list)
    cm_class = defaultdict(list)
    sample_class = defaultdict(lambda: defaultdict(str))

    # Initialize the ambiguous-hit count and first-file flag
    num_amb_hit = 0
    is_fist_file = 1

    # Read all BLAST result files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # Skip non-result files
                continue

            # Path to the BLAST result file
            full_path = os.path.join(dirpath, filename)

            if os.path.getsize(full_path) == 0:
                print("filesize 0")
                continue

            # Annotate and save viral contigs
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each

            # Store contig counts by confusion-matrix group
            cm_group = "external"
            cm_hits[cm_group].append(num_viral_black)
            cm_props[cm_group].append(num_viral_black / num_all_contigs)
    
            # Classify samples using the contig-count threshold
            if num_viral_black >= infection_th:
                cm_class[cm_group].append("1")
            else:
                cm_class[cm_group].append("0")

            # Combine sample results
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])

            # Create the sample summary table
            sample = full_path.split('/')[-1].split(".")[0].split("_")[0]
            if num_viral_black >= infection_th:
                sample_class[sample]["viral_contig"] = "1"
            else:
                sample_class[sample]["viral_contig"] = "0"
            sample_class[sample]["cm"] = cm_group

    print("Virus ambiguous hit:", num_amb_hit)
    print("=======")

    for k, v in cm_hits.items():
        print(k, np.mean(v))
    for k, v in cm_props.items():
        print(k, f"{np.mean(v):.2}")

    print("=======")
    return cm_hits, cm_props, cm_class, sample_class, df_all


def collect_from_tsv(tsv_file, output_file):
    """
    Collect sequences listed in a TSV into a single multi-FASTA file.

    Parameters:
        tsv_file (str): Path to TSV file with columns [path, qseqid]
        output_file (str): Path to the output FASTA file
    """
    df = pd.read_table(tsv_file, sep="\t")
    with open(output_file, "w") as out_f:
        for fasta_file in sorted(list(set(df["path"].to_list()))):
            record_dict = SeqIO.index(fasta_file, "fasta")
            contigs = df.loc[df["path"] == fasta_file, "qseqid"].to_list()
            for contig in contigs:
                if contig in record_dict:
                    SeqIO.write(record_dict[contig], out_f, "fasta")
                else:
                    print(f"[Warning] {contig} not found in {fasta_file}")


def get_seq_from_fasta(fasta_file, contig_name):
    """
    Return the sequence of a named contig from a FASTA file.

    Parameters:
        fasta_file (str): Path to the FASTA file
        contig_name (str): Contig ID (matches the part after '>' up to first space)

    Returns:
        str: Sequence as a string, or None if not found
    """
    record_dict = SeqIO.index(fasta_file, "fasta")
    if contig_name in record_dict:
        return record_dict[contig_name].seq
    else:
        return None


def asign_confusion_matrix(true_label, ml_label):
    res = np.nan
    if true_label == 1 and ml_label == 1:
        res = "TP"
    if true_label == 1 and ml_label == 0:
        res = "FN"
    if true_label == 0 and ml_label == 1:
        res = "FP"
    if true_label == 0 and ml_label == 0:
        res = "TN"
    if res != res:
        print("failed to assign cm group.")
    return res

# TaxonKit

In [4]:
# Collect taxids from all samples
taxids = get_all_taxids(path_in, path_out + "taxids.txt")

34114 taxids in total


Run `Fig4_table_taxonkit.sh` in the taxonkit environment.

# Identify phages

In [5]:
# Read TaxonKit output
taxinfo = pd.read_table(path_out + "taxids_long.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# Split the lineage into taxonomic ranks
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# Check missing values
print("#NaN:", taxinfo.isna().sum().sum())
# Fill missing values
taxinfo = taxinfo.fillna("-")

# Check table dimensions
print("taxonkit output original size:", taxinfo.shape[0])

# Identify phage taxa
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. Unclassified viral species containing "phage"
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# Summarize phage orders
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# Check table dimensions
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# Save the annotation table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# Save the phage taxid list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# Preview results
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 18
taxonkit output original size: 34114
phage order: ['Crassvirales', 'unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 261
phage: 108
 * not Caudoviricetes: 3
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae
15512,1892901,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,ssRNA phage DC,1,0,1,0,0,0
23991,278008,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured phage,1,0,1,0,0,0
26413,38018,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacteriophage sp.,1,0,1,0,0,0


# Add taxonomy

In [6]:
# Read TaxonKit output
taxid_in = path_out + "taxids_lineage.txt"
lineage_info = pd.read_table(taxid_in, header=None)
lineage_info.columns = ["taxid", "domain", "fgs"]

# Check missing values
print("#NaN beginning:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Add family, genus and species columns
lineage_info["taxid"] = lineage_info["taxid"].astype(str)
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)  
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)  
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)  

# Label phage taxa as phage_group
phage_ids = []
with open(path_out + "phage_list.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# Check missing values
print("#NaN end:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# Fill missing values
lineage_info = lineage_info.fillna("-")

print(f"{lineage_info.shape[0]} taxids in total\n")

# Preview hits from each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# Preview results
lineage_info

#NaN beginning: 4
        taxid domain  fgs
3288  1230063    NaN  NaN
3291  1230068    NaN  NaN

#NaN end: 10
        domain  fgs family genus species
taxid                                   
1230063    NaN  NaN    NaN   NaN     NaN
1230068    NaN  NaN    NaN   NaN     NaN

34114 taxids in total

All domain: {'Archaea', 'Eukaryota', 'phage_group', 'Bacteria', '-', 'unclassified other entries domain', 'unclassified Viruses domain', 'unclassified unclassified entries domain'} 

##### Archaea ####
size: 91
          domain                           species
taxid                                             
1070774  Archaea             Halonotius sp. J07HN4
1072681  Archaea  Candidatus Haloredivivus sp. G17
1185654  Archaea               Pyrococcus furiosus

##### Eukaryota ####
size: 3427
            domain                      species
taxid                                          
100035   Eukaryota  Massariosphaeria phaeospora
1000413  Eukaryota              Acer yangbiense
100182   Eu

,domain,fgs,family,genus,species
taxid,,,,,
1000289,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000293,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000294,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000295,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000296,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
...,...,...,...,...,...
999810,Eukaryota,Sclerotiniaceae;Botrytis;Botrytis cinerea,Sclerotiniaceae,Botrytis,Botrytis cinerea
999849,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
999883,unclassified Viruses domain,Marseilleviridae;unclassified Marseilleviridae...,Marseilleviridae,unclassified Marseilleviridae genus,Lausannevirus


# Extract viral top hits

In [7]:
# Extract viral top hits
infection_threshod = 1
cm_hits, cm_props, cm_class, sample_class, df_all = count_viral_contigs(path_in, path_out, lineage_info, blacklist_path, infection_threshod)

# Create the viral contig table
df = df_all.copy().reset_index()
# Check sample IDs
for i in df.index:
    qseqid = df.loc[i, "qseqid"]
    if type(qseqid)==int:
        continue
    else:
        if qseqid.split("_")[0]!=df.loc[i, "file_name"]:
           print(df.loc[i, :])
print("done")
df.columns = df.columns.map(lambda x: x.replace("file_name", "sample"))

# Add confusion-matrix labels
sampledf = pd.DataFrame(sample_class).T
df = df.merge(sampledf, left_on = "sample", right_index=True, how="left")

# Add viral contig counts per sample
virus_hit_count = pd.DataFrame(df.value_counts("sample"))
df = df.merge(virus_hit_count, on="sample", how="left")
df = df.sort_values("sample")
df = df.sort_values("count", ascending=False)

# Check missing values
print("#NaN:", df.isna().sum().sum())

# Save results
df.to_csv(path_out + "virus_all.txt", sep="\t")

Virus ambiguous hit: 9006
external 5.720734506503443
external 0.0094
done
#NaN: 0


# BLASTn

In [8]:
# Prepare BLASTn inputs
# Create a table of contig IDs and FASTA paths
df = pd.read_table(path_out + "virus_all.txt", index_col=0)
df = df.drop("viral_contig", axis=1)
df_fasta = df[["qseqid", "cm", "sample"]].copy()
for idx in df_fasta.index:
    df_fasta.loc[idx, "path"] = f"{path_in_fasta}{df_fasta.loc[idx, "sample"]}.viral_hit_contigs.fa"
df_fasta = df_fasta[["path", "qseqid"]]
# Save the annotation table
df_fasta.to_csv(path_out + "virus_contig_path.txt", sep="\t", index=False)
# Collect viral contigs into a FASTA file
collect_from_tsv(path_out + "virus_contig_path.txt", path_out + "virus_hit_all_contigs.fasta")

Run `Fig4_table_blastn.sh` on the HPC and download its output to `path_in_blastn` before continuing.

# TaxonKit for BLASTn hits

In [9]:
# Prepare TaxonKit inputs
# Taxid list for TaxonKit
takonkit_path_2 = path_out + "taxids_blastn.txt"

# Initialize the taxid set
taxids = set()

# Read BLASTn hits
col_name= [
    "qseqid", "sseqid", "pident", "length", 
    "mismatch", "gapopen", "qstart", "qend", 
    "sstart", "send", "evalue", "bitscore", 
    "qframe", "staxids", "stitle", "qlen", 
    "slen", "qcovs", "qcovhsp"
]

blastn = pd.read_table(path_in_blastn, header=None, dtype={13:str})  # Read taxids as strings
blastn.columns = col_name
print("blastn output original size:", blastn.shape)
print("#NaN:", blastn.isna().sum().sum())

# Sort hits by bit score
blastn_sorted = blastn.sort_values("bitscore", ascending=False)

# Remove hits without taxids
blastn_sorted = blastn_sorted.dropna(subset=["staxids"])

# Keep the top hit for each query
blastn_uniq = blastn_sorted.groupby("qseqid").first()
print("blastn output size after choosing tophits:", blastn_uniq.shape)

# Collect taxids
for line_taxid in blastn_uniq["staxids"].to_list():
    if (line_taxid is None) or (line_taxid!=line_taxid):
        continue
    raw_taxids = line_taxid.split(";")
    for i in range(len(raw_taxids)):
        taxids.add(raw_taxids[i])

# Write one taxid per line
with open(takonkit_path_2, "w") as f:
    for taxid in taxids:
        f.write(taxid + "\n")

print(f"{len(taxids)} taxids in total")

# Preview results
blastn_uniq.head(10)

blastn output original size: (6914676, 19)
#NaN: 0
blastn output size after choosing tophits: (20808, 18)
1832 taxids in total


,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qframe,staxids,stitle,qlen,slen,qcovs,qcovhsp
qseqid,,,,,,,,,,,,,,,,,,
102179,gi|2196303737|gb|MZ679293.1|,91.579,285,24,0,1,285,6037,5753,3.980000e-105,394.0,1,1930506,MAG: Sicinivirus sp. isolate R22-k141_395554 p...,285,6092,100,100
1042341,gi|2171934023|gb|OM079140.1|,100.000,213,0,0,1,213,8015,7803,2.830000e-105,394.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,213,29799,100,100
1060688,gi|2493070242|gb|OQ852598.1|,99.892,928,1,0,1,928,22073,23000,0.000000e+00,1709.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,928,29706,100,100
1088902,gi|2240251053|gb|MT978647.1|,77.665,197,42,2,19,214,299,104,1.920000e-22,119.0,1,10788,Canine parvovirus strain CPV-2c/Myanmar/RS-45/...,214,1746,92,92
113348,gi|2196303729|gb|MZ679278.1|,96.495,428,15,0,1,428,3601,4028,0.000000e+00,708.0,1,1930506,MAG: Sicinivirus sp. isolate 363R-k141_188215 ...,428,5304,100,100
1154076,gi|2305169510|gb|OP468871.1|,100.000,478,0,0,1,478,24866,24389,0.000000e+00,883.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,478,29718,100,100
117398,gi|2918479373|gb|PV093045.1|,91.541,863,73,0,1,863,6413,7275,0.000000e+00,1190.0,1,1930506,"Sicinivirus sp. strain WHM13, partial genome",863,9854,100,100
1190696,gi|2112847088|emb|OU815371.1|,100.000,407,0,0,1,407,13262,12856,0.000000e+00,752.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,407,29899,100,100
119917,gi|408684319|dbj|AB753441.1|,100.000,1064,0,0,1,1064,1608,2671,0.000000e+00,1965.0,1,3052731,"Respirovirus muris P, N genes for phosphoprote...",1064,3431,100,100


Run `Fig4_table_taxonkit_blastn.sh` in the taxonkit environment.

# Identify phages in BLASTn hits

In [10]:
# Identify phages
# Read TaxonKit output
taxinfo = pd.read_table(path_out + "taxids_long_blastn.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# Split the lineage into taxonomic ranks
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# Check missing values
print("#NaN:", taxinfo.isna().sum().sum())
# Fill missing values
taxinfo = taxinfo.fillna("-")
# Check table dimensions
print("taxonkit result size:", taxinfo.shape[0])

# Identify phage taxa
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. Unclassified viral species containing "phage"
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# Summarize phage orders
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# Check table dimensions
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# Save the annotation table
taxinfo.to_csv(path_out + "taxids_phage_blastn.tsv", sep="\t")

# Save the phage taxid list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list_blastn.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# Preview results
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 9
taxonkit result size: 1832
phage order: []
Caudoviricetes: 1
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae


In [11]:
# Add taxonomic annotations to the taxid list
# Input file path
taxid_in = path_out + "taxids_lineage_blastn.txt"

# Read the lineage table
lineage_info = pd.read_table(taxid_in, header=None, dtype={0:str})
lineage_info.columns = ["taxid", "domain", "fgs"]

# Set the fallback taxonomy for taxid 3434951
lineage_info.loc[lineage_info["taxid"]=="3434951", "domain"] = "unclassified Viruses domain"
lineage_info.loc[lineage_info["taxid"]=="3434951", "fgs"] = "Picobirnaviridae;Orthopicobirnavirus;unclassified Orthopicobirnavirus"

# Check missing values
print("#NaN:", lineage_info.isna().sum().sum())

# Add family, genus and species columns
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)


# Label phage taxa as phage_group
phage_ids = []
with open(path_out + "phage_list_blastn.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# Fill missing values
lineage_info = lineage_info.fillna("-")

# Check table dimensions
print(f"{lineage_info.shape[0]} taxids in total\n")

# Preview hits from each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# Preview results
lineage_info

#NaN: 0
1832 taxids in total

All domain: {'Eukaryota', 'phage_group', 'Bacteria', 'unclassified other entries domain', 'unclassified Viruses domain'} 

##### Eukaryota ####
size: 37
          domain                species
taxid                                  
9315   Eukaryota   Notamacropus eugenii
94328  Eukaryota    Zingiber officinale
9986   Eukaryota  Oryctolagus cuniculus

##### phage_group ####
size: 1
              domain             species
taxid                                   
2832643  phage_group  Caudoviricetes sp.

##### Bacteria ####
size: 1
           domain                    species
taxid                                       
1582221  Bacteria  uncultured Kistimonas sp.

##### unclassified other entries domain ####
size: 9
                                    domain  \
taxid                                        
161364   unclassified other entries domain   
3390492  unclassified other entries domain   
3390493  unclassified other entries domain   

             

,domain,fgs,family,genus,species
taxid,,,,,
1589307,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1910011,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1140162,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
42785,unclassified Viruses domain,Picornaviridae;Enterovirus;Enterovirus A,Picornaviridae,Enterovirus,Enterovirus A
2026025,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
...,...,...,...,...,...
2810457,unclassified Viruses domain,Astroviridae;Avastrovirus;Goose astrovirus 2,Astroviridae,Avastrovirus,Goose astrovirus 2
3130793,unclassified Viruses domain,Dicistroviridae;unclassified Dicistroviridae g...,Dicistroviridae,unclassified Dicistroviridae genus,Ripishoj virus
1841048,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae


In [12]:
# Annotate BLASTn top hits
# Read BLASTn hits
blastn_tax = blastn_uniq.copy()
blastn_tax["staxids_lst"] = blastn_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
blastn_tax["name_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
blastn_tax["group_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

# Check taxonomic assignments
num_ambiguous_hit = 0
for idx in blastn_uniq.index:
    groups = blastn_tax.loc[idx, "group_lst"]
    names = blastn_tax.loc[idx, "name_lst"]
    # Count hits spanning viral and non-viral taxonomic groups
    if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
        num_ambiguous_hit += 1
        print("#### Different root ####")
        print(groups)
        print(names)
        print()
print("Num ambiguou hits:", num_ambiguous_hit)

# Assign family, genus and species from the first taxid
blastn_tax["taxid"] = blastn_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  
blastn_tax["family"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "family"] if((x is not None) and (x==x)) else x)
blastn_tax["genus"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "genus"] if((x is not None) and (x==x)) else x)
blastn_tax["species"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "species"] if((x is not None) and (x==x)) else x)

# Check missing values
print("#NaN:", blastn_tax.isna().sum().sum())

# Mark hits as viral only when all taxonomic groups are viral
blastn_tax["is_virus"] = blastn_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

blastn_tax.columns = blastn_tax.columns.map(lambda x: x + "_blastn")  # Add the _blastn suffix to annotation columns

# Save results
blastn_tax.to_csv(path_out + "blastn_result_top_hit.tsv", sep="\t")

# Preview results
print("Size after adding taxonomy:", blastn_tax.shape)
blastn_tax.head(10)

Num ambiguou hits: 0
#NaN: 0
Size after adding taxonomy: (20808, 26)


,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,evalue_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
qseqid,,,,,,,,,,,,,,,,,,,,,
102179,gi|2196303737|gb|MZ679293.1|,91.579,285,24,0,1,285,6037,5753,3.980000e-105,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1042341,gi|2171934023|gb|OM079140.1|,100.000,213,0,0,1,213,8015,7803,2.830000e-105,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
1060688,gi|2493070242|gb|OQ852598.1|,99.892,928,1,0,1,928,22073,23000,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
1088902,gi|2240251053|gb|MT978647.1|,77.665,197,42,2,19,214,299,104,1.920000e-22,...,92,92,[10788],[Parvoviridae;Protoparvovirus;Protoparvovirus ...,[unclassified Viruses domain],10788,Parvoviridae,Protoparvovirus,Protoparvovirus carnivoran1,1
113348,gi|2196303729|gb|MZ679278.1|,96.495,428,15,0,1,428,3601,4028,0.000000e+00,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1154076,gi|2305169510|gb|OP468871.1|,100.000,478,0,0,1,478,24866,24389,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
117398,gi|2918479373|gb|PV093045.1|,91.541,863,73,0,1,863,6413,7275,0.000000e+00,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1190696,gi|2112847088|emb|OU815371.1|,100.000,407,0,0,1,407,13262,12856,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
119917,gi|408684319|dbj|AB753441.1|,100.000,1064,0,0,1,1064,1608,2671,0.000000e+00,...,100,100,[3052731],[Paramyxoviridae;Respirovirus;Respirovirus muris],[unclassified Viruses domain],3052731,Paramyxoviridae,Respirovirus,Respirovirus muris,1


# Save tables

In [13]:
# Select samples at random
# Read sample information
sample_info = pd.read_table(sradata_path)
sample_info_analyzed = sample_info.loc[sample_info["diamond_blastx"] == 1, :].copy()
# Set the sample size
sampe_size = 1000 - sample_info_analyzed.loc[((sample_info_analyzed["Negative"]==3)|(sample_info_analyzed["Negative"]==4)),:].shape[0]
random.seed(42) 
subset_lst = random.sample(sample_info_analyzed.loc[sample_info_analyzed["Negative"]==5, "ID"].to_list(), sampe_size)
sample_info_analyzed["random_sampling"] = sample_info_analyzed.apply(
    lambda x: 1 if x["Negative"] != 5 else (1 if x["ID"] in subset_lst else 0),
    axis=1
)
# Add LOGAN metadata
sra_meta = pd.read_table(srameta_path)
sra_meta.columns = sra_meta.columns.map(lambda x: x + "_host" if x in ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'] else x)
# Merge tables
combined = sample_info_analyzed.merge(sra_meta, on="ID", how="left")
# Assign predicted labels
combined["ML_label"] = combined["Positive"].apply(lambda x: 1 if x >= 3 else 0)
# Assign ground-truth labels
combined["True_label"] = combined["ISG_level"].apply(lambda x: 1 if x in ["high", "both"] else (0 if x in ["Negative", "low"] else np.nan))
# Assign confusion-matrix labels
combined["cm"] = combined.apply(lambda x: asign_confusion_matrix(x["True_label"], x["ML_label"]), axis=1)
# Save results
combined.to_csv(path_out + "cm_assignment.tsv", sep="\t")
combined.head(3)

,ID,Negative,Positive,diamond_blastx,random_sampling,Host_species,BioProject_ID,species_host,tax_id,kingdom_host,...,class_host,order_host,family_host,genus_host,nucleotide_DNA,nucleotide_RNA,ISG_level,ML_label,True_label,cm
0,ERR11219020,0,5,1,1,Sus_scrofa,PRJEB61259,Sus_scrofa,9823,Eukaryota,...,Mammalia,Artiodactyla,Suidae,Sus,Positive,Negative,high,1,1,TP
1,ERR11219054,0,5,1,1,Sus_scrofa,PRJEB61259,Sus_scrofa,9823,Eukaryota,...,Mammalia,Artiodactyla,Suidae,Sus,Negative,Positive,high,1,1,TP
2,ERR11630589,2,3,1,1,Gallus_gallus,PRJEB63475,Gallus_gallus,9031,Eukaryota,...,Aves,Galliformes,Phasianidae,Gallus,Negative,Negative,Negative,1,0,FP


In [14]:
# Merge BLASTx and BLASTn annotations
# Read BLASTx results
df_all = pd.read_table(path_out + "virus_all.txt", index_col=0)

# Read BLASTn hits
df_blastn = pd.read_table(path_out + "blastn_result_top_hit.tsv")
print("df_blastn shape:", df_blastn.shape)

# Merge BLASTx and BLASTn results
df_all_blastn = df_all.merge(df_blastn, on="qseqid", how="left")
print("df_all_blastn shape:", df_all_blastn.shape)

# Select samples at random
cm_info = pd.read_table(path_out + "cm_assignment.tsv", index_col=0)
df_all_blastn = df_all_blastn.drop("cm", axis=1)
df_all_blastn = df_all_blastn.merge(cm_info[["ID", "random_sampling", "cm", "ML_label"]], left_on="sample", right_on="ID", how="left")
df_all_sampled = df_all_blastn.loc[df_all_blastn["random_sampling"]==1, :].copy()
print("size of df_all_sampled:", df_all_sampled.shape)

# Add query sequences
df_all_sampled["query_seq"] = df_all_sampled.apply(lambda row: get_seq_from_fasta(f"{path_in_fasta}{row["sample"]}.viral_hit_contigs.fa", row["qseqid"]), axis=1)
df_all_sampled["qseqid_formatted"] = df_all_sampled.apply(lambda x: x["sample"] + "_" + x["qseqid"] if not(x["qseqid"].startswith("E") or x["qseqid"].startswith("S")) else x["qseqid"], axis=1)

# Save results
df_all_sampled.to_csv(path_out + "virus_all.blastn.randomsampled.tsv", sep="\t")
# Preview results
df_all_sampled

df_blastn shape: (20808, 27)
df_all_blastn shape: (22431, 57)
size of df_all_sampled: (22414, 60)


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,family_blastn,genus_blastn,species_blastn,is_virus_blastn,ID,random_sampling,cm,ML_label,query_seq,qseqid_formatted
0,ERR9896241_167993,AWK02683.1,96.3,80,3,0,2,241,984,1063,...,Picornaviridae,Cardiovirus,Cardiovirus C,1.0,ERR9896241,1,TP,1,"(A, G, T, T, C, T, C, T, T, C, T, T, T, G, A, ...",ERR9896241_167993
1,ERR9896241_156505,ULF99621.1,98.6,73,1,0,219,1,319,391,...,Picornaviridae,Megrivirus,Megrivirus sp.,1.0,ERR9896241,1,TP,1,"(A, A, C, C, T, G, G, G, T, T, G, C, T, G, C, ...",ERR9896241_156505
2,ERR9896241_109066,ULF99621.1,92.1,101,8,0,303,1,564,664,...,Picornaviridae,Megrivirus,Megrivirus sp.,1.0,ERR9896241,1,TP,1,"(A, A, C, A, C, C, G, G, G, G, G, C, T, C, T, ...",ERR9896241_109066
3,ERR9896241_120057,ULF99621.1,89.6,77,8,0,1,231,962,1038,...,Picornaviridae,Megrivirus,Megrivirus sp.,1.0,ERR9896241,1,TP,1,"(G, A, G, C, G, G, A, C, C, A, T, G, T, A, T, ...",ERR9896241_120057
4,ERR9896241_122125,APG76091.1,95.9,440,18,0,2,1321,172,611,...,unclassified Riboviria family,unclassified Riboviria genus,Hubei noda-like virus 11,1.0,ERR9896241,1,TP,1,"(G, T, T, G, A, C, C, G, A, T, A, A, A, C, A, ...",ERR9896241_122125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22426,SRR27998875_30809,YP_009055046.1,98.9,88,1,0,1,264,319,406,...,Picornaviridae,Megrivirus,Megrivirus sp.,1.0,SRR27998875,1,FP,1,"(A, T, T, G, C, T, C, T, G, A, A, A, G, T, T, ...",SRR27998875_30809
22427,SRR27998889_13132,QKE55050.1,98.4,61,1,0,184,2,1959,2019,...,Picornaviridae,unclassified Picornaviridae genus,Picornaviridae sp.,1.0,SRR27998889,1,FP,1,"(C, G, G, T, A, A, T, T, G, T, A, A, A, A, G, ...",SRR27998889_13132
22428,SRR28047234_15934,AWC68490.1,56.5,85,35,1,3,251,164,248,...,Petauridae,Petaurus,Petaurus breviceps,0.0,SRR28047234,1,FP,1,"(A, A, C, T, C, C, T, G, A, A, T, C, T, T, C, ...",SRR28047234_15934
22429,SRR28047243_57124,AWC68490.1,57.7,52,22,0,158,3,180,231,...,Petauridae,Petaurus,Petaurus breviceps,0.0,SRR28047243,1,FP,1,"(C, C, A, A, G, G, C, T, C, T, T, G, A, C, T, ...",SRR28047243_57124


# Add SRA metadata

In [15]:
# Add SRA, BioProject and prediction metadata to the sampled contigs
# Read the saved sample-selection results
ml_out = pd.read_table(sradata_path, dtype={"ID": str})
ml_out["ML_>=3_hit"] = (ml_out["Positive"] >= 3).astype(int)

sra_meta = pd.read_table(
    srameta_path, dtype={"ID": str, "BioProject_ID": str, "tax_id": "Int64"},
)
host_ranks = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
sra_meta = sra_meta.rename(columns={rank: rank + "_host" for rank in host_ranks})

# Check BioProject fields for conflicts before selecting one row per project
project_columns = ["Project_Id", "Project_Acc", "Project_Title", "Project_Description"]
sra_meta2 = pd.read_table(
    f"{basedir}/data/SRA_data/output_table3_with_infection2.txt",
    dtype={"Project_Id": "Int64", "Project_Acc": str},
)[project_columns]
if sra_meta2["Project_Id"].isna().any():
    raise ValueError("BioProject metadata contains missing Project_Id values.")
project_conflicts = sra_meta2.groupby("Project_Id")[project_columns[1:]].nunique(dropna=False).gt(1)
if project_conflicts.any(axis=1).any():
    conflict_ids = project_conflicts.index[project_conflicts.any(axis=1)].tolist()
    raise ValueError(f"Conflicting BioProject rows; groupby.first() would hide differences: {conflict_ids}")
sra_meta2 = sra_meta2.groupby("Project_Id").first().reset_index()

# Reuse CM labels and sampling fields from the saved BLAST results
df_virus = pd.read_table(
    path_out + "virus_all.blastn.randomsampled.tsv", index_col=0,
    dtype={"qseqid": str, "sample": str, "ID": str,
           "taxid_blastx": "Int64", "taxid_blastn": "Int64", "is_virus_blastn": "Int64"},
)
if df_virus.duplicated(["sample", "qseqid"]).any():
    raise ValueError("Duplicated (sample, qseqid) keys in sampled BLAST results.")
if df_virus["ID"].isna().any() or df_virus["ID"].ne(df_virus["sample"]).any():
    raise ValueError("Sample and existing Run ID disagree in sampled BLAST results.")
cm_info = pd.read_table(path_out + "cm_assignment.tsv", index_col=0, dtype={"ID": str})
cm_by_run = cm_info.set_index("ID", verify_integrity=True)
for column in ["cm", "random_sampling", "ML_label"]:
    expected = df_virus["ID"].map(cm_by_run[column])
    if df_virus[column].ne(expected).any():
        raise ValueError(f"Sampled results disagree with cm_assignment.tsv: {column}")
if not df_virus["random_sampling"].eq(1).all():
    raise ValueError("Input contains rows outside the selected random sample.")

# Join SRA, BioProject and prediction metadata by sample and project ID
df_metadata = df_virus.drop(columns=["viral_contig", "ID"]).merge(
    sra_meta, left_on="sample", right_on="ID", how="left", validate="many_to_one",
)
df_metadata = df_metadata.merge(
    sra_meta2, left_on="BioProject_ID", right_on="Project_Acc",
    how="left", validate="many_to_one",
)
df_metadata["Project_Id"] = df_metadata["Project_Id"].astype("Int64")
df_metadata = df_metadata.merge(
    ml_out[["ID", "ML_>=3_hit"]], on="ID", how="left", validate="many_to_one",
)
if df_metadata["ID"].isna().any():
    missing_runs = df_metadata.loc[df_metadata["ID"].isna(), "sample"].unique().tolist()
    raise ValueError(f"Run IDs missing from SRA metadata: {missing_runs}")
if df_metadata["ML_>=3_hit"].ne(df_metadata["ML_label"]).any():
    raise ValueError("ML_>=3_hit disagrees with the input ML_label.")
df_metadata = df_metadata.drop(columns=["ML_label"])
print("Contigs:", len(df_metadata), "Runs:", df_metadata["ID"].nunique())
print("Missing metadata (Project details may be absent in the original project catalogue):")
print(df_metadata[["ID", "BioProject_ID", "Host_species", "tax_id", "cm", "Project_Acc"]].isna().sum())


Contigs: 22414 Runs: 1707
Missing metadata (Project details may be absent in the original project catalogue):
ID                   0
BioProject_ID        0
Host_species         0
tax_id               0
cm                   0
Project_Acc      18232
dtype: int64


In [16]:
# Select the metadata columns in the required order
column_order = [
    'Project_Id', 'Project_Acc', 'Project_Title', 'Project_Description',
    'ID', 'Host_species', 'BioProject_ID', 'species_host',
    'tax_id', 'kingdom_host', 'phylum_host', 'class_host',
    'order_host', 'family_host', 'genus_host', 'nucleotide_DNA',
    'nucleotide_RNA', 'ISG_level', 'ML_>=3_hit', 'qseqid',
    'qseqid_formatted', 'sseqid', 'pident', 'length',
    'mismatch', 'gapopen', 'qstart', 'qend',
    'sstart', 'send', 'evalue', 'bitscore',
    'qframe', 'staxids', 'stitle', 'qlen',
    'slen', 'qcovhsp', 'scovhsp', 'staxids_lst',
    'name_lst', 'group_lst', 'taxid_blastx', 'family',
    'genus', 'species', 'is_virus', 'sseqid_blastn',
    'pident_blastn', 'length_blastn', 'mismatch_blastn', 'gapopen_blastn',
    'qstart_blastn', 'qend_blastn', 'sstart_blastn', 'send_blastn',
    'evalue_blastn', 'bitscore_blastn', 'qframe_blastn', 'staxids_blastn',
    'stitle_blastn', 'qlen_blastn', 'slen_blastn', 'qcovs_blastn',
    'qcovhsp_blastn', 'staxids_lst_blastn', 'name_lst_blastn', 'group_lst_blastn',
    'taxid_blastn', 'family_blastn', 'genus_blastn', 'species_blastn',
    'is_virus_blastn', 'sample', 'cm', 'count',
    'random_sampling', 'query_seq',
]
df_metadata = df_metadata[column_order]

metadata_output_path = os.path.join(path_out, "virus_all_3921.meta.txt")
metadata_text = df_metadata.to_csv(sep="\t")
if os.path.exists(metadata_output_path):
    with open(metadata_output_path, encoding="utf-8") as stream:
        existing_text = stream.read()
    if existing_text != metadata_text:
        raise FileExistsError(f"Existing result differs; choose a new metadata output path: {metadata_output_path}")
    print("Existing identical metadata retained:", metadata_output_path)
else:
    with open(metadata_output_path, "x", encoding="utf-8", newline="") as stream:
        stream.write(metadata_text)
    print("Saved new metadata:", metadata_output_path)
print("Shape:", df_metadata.shape)
df_metadata.head()


Saved new metadata: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig4/virus_all_3921.meta.txt
Shape: (22414, 78)


,Project_Id,Project_Acc,Project_Title,Project_Description,ID,Host_species,BioProject_ID,species_host,tax_id,kingdom_host,...,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn,sample,cm,count,random_sampling,query_seq
0,996436,PRJEB53828,"Virus detection in wild rats, Rattus norvegicus",Rats were trapped between June 2020 and Novemb...,ERR9896241,Rattus_norvegicus,PRJEB53828,Rattus_norvegicus,10116,Eukaryota,...,1821751,Picornaviridae,Cardiovirus,Cardiovirus C,1,ERR9896241,TP,229,1,AGTTCTCTTCTTTGAACGCCCCTATGGCTATAAGATGCAGTATAGG...
1,996436,PRJEB53828,"Virus detection in wild rats, Rattus norvegicus",Rats were trapped between June 2020 and Novemb...,ERR9896241,Rattus_norvegicus,PRJEB53828,Rattus_norvegicus,10116,Eukaryota,...,2809560,Picornaviridae,Megrivirus,Megrivirus sp.,1,ERR9896241,TP,229,1,AACCTGGGTTGCTGCAGCCACAGCACCATCCCTTGTTGTCGAGCCA...
2,996436,PRJEB53828,"Virus detection in wild rats, Rattus norvegicus",Rats were trapped between June 2020 and Novemb...,ERR9896241,Rattus_norvegicus,PRJEB53828,Rattus_norvegicus,10116,Eukaryota,...,2809560,Picornaviridae,Megrivirus,Megrivirus sp.,1,ERR9896241,TP,229,1,AACACCGGGGGCTCTATACGCAACAATCCAGTCACTACTTCCTGGC...
3,996436,PRJEB53828,"Virus detection in wild rats, Rattus norvegicus",Rats were trapped between June 2020 and Novemb...,ERR9896241,Rattus_norvegicus,PRJEB53828,Rattus_norvegicus,10116,Eukaryota,...,2809560,Picornaviridae,Megrivirus,Megrivirus sp.,1,ERR9896241,TP,229,1,GAGCGGACCATGTATGATAGGCAGGTTAGAGTGTATACACCTCTGA...
4,996436,PRJEB53828,"Virus detection in wild rats, Rattus norvegicus",Rats were trapped between June 2020 and Novemb...,ERR9896241,Rattus_norvegicus,PRJEB53828,Rattus_norvegicus,10116,Eukaryota,...,1922967,unclassified Riboviria family,unclassified Riboviria genus,Hubei noda-like virus 11,1,ERR9896241,TP,229,1,GTTGACCGATAAACACGTGTTGATGTTCATTGACGTTGACTACTAT...
